# SAM2 Decoder Fine-Tuning

Fine-tunes the SAM2 mask decoder on the Prieur et al. boulder dataset.
The image encoder (Hiera ViT) is frozen; only the prompt encoder and mask decoder are trained.

Box prompts come from ground-truth polygon bounding boxes — same prompt format used at inference.

In [ ]:
import sys
sys.path.insert(0, "/scratch/users/cayleigh/YOLOv8-BeyondEarth/src")
if "/home/users/cayleigh/BoulderNet/YOLOv8-BeyondEarth/src" in sys.path:
    sys.path.remove("/home/users/cayleigh/BoulderNet/YOLOv8-BeyondEarth/src")

import typing_extensions
if not hasattr(typing_extensions, "TypeIs"):
    typing_extensions.TypeIs = typing_extensions.TypeGuard

import torch
import torch.nn.functional as F
import numpy as np
import geopandas as gpd
import rasterio
import rasterio.mask
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
sam2_checkpoint = Path("/scratch/users/cayleigh/checkpoints/sam2.1_hiera_small.pt")
sam2_config     = "configs/sam2.1/sam2.1_hiera_s.yaml"
dataset_dir     = Path("/scratch/users/cayleigh/Apr2023-Mars-Moon-Earth-mask-5px/preprocessing")
ckpt_out_dir    = Path("/scratch/users/cayleigh/sam2_finetuned")
ckpt_out_dir.mkdir(parents=True, exist_ok=True)

# ── Training hyperparameters ──────────────────────────────────────────────────
device      = "cuda:0"
epochs      = 10
lr          = 1e-4
min_area_px = 36    # minimum GT instance area in pixels (6×6)

In [ ]:
sam2_model = build_sam2(sam2_config, sam2_checkpoint, device=device)

# Freeze image encoder only
for name, param in sam2_model.named_parameters():
    param.requires_grad_(False if "image_encoder" in name else True)

n_train  = sum(p.numel() for p in sam2_model.parameters() if p.requires_grad)
n_frozen = sum(p.numel() for p in sam2_model.parameters() if not p.requires_grad)
print(f"Trainable: {n_train/1e6:.1f}M  |  Frozen: {n_frozen/1e6:.1f}M")

predictor = SAM2ImagePredictor(sam2_model)
optimizer = torch.optim.Adam(
    [p for p in sam2_model.parameters() if p.requires_grad], lr=lr
)

In [ ]:
def load_tile_instances(img_tif, label_shp):
    """
    Returns list of (img_rgb, box_np, gt_mask) per GT boulder in the tile.
    img_rgb:  H×W×3 uint8 (grayscale repeated 3×, as SAM2 expects RGB)
    box_np:   [x1, y1, x2, y2] float32 in pixel coords
    gt_mask:  H×W bool
    """
    gdf = gpd.read_file(label_shp)
    if gdf.empty:
        return []

    with rasterio.open(img_tif) as src:
        raw = src.read(1).astype(np.float32)
        norm = (raw / raw.max() * 255).clip(0, 255).astype(np.uint8) if raw.max() > 0 else raw.astype(np.uint8)
        img_rgb = np.stack([norm, norm, norm], axis=-1)

        instances = []
        for _, row in gdf.iterrows():
            out, _ = rasterio.mask.mask(src, [row.geometry], crop=False, all_touched=False)
            gt_mask = (out[0] > 0)
            if gt_mask.sum() < min_area_px:
                continue
            ys, xs = np.where(gt_mask)
            box_np = np.array([float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())])
            instances.append((img_rgb, box_np, gt_mask))

    return instances


def gather_tiles(split):
    img_dir   = dataset_dir / split / "images"
    label_dir = dataset_dir / split / "labels"
    tiles = []
    for img_tif in sorted(img_dir.glob("*_image.tif")):
        shp = label_dir / img_tif.name.replace("_image.tif", "_mask.shp")
        if shp.exists():
            tiles.append((img_tif, shp))
    return tiles


train_tiles = gather_tiles("train")
val_tiles   = gather_tiles("validation")
print(f"Train tiles: {len(train_tiles)}  |  Val tiles: {len(val_tiles)}")

In [ ]:
def seg_loss(logits, gt_mask):
    """BCE + Dice loss on logits."""
    gt   = torch.as_tensor(gt_mask, dtype=torch.float32, device=device)
    bce  = F.binary_cross_entropy_with_logits(logits, gt)
    pred = torch.sigmoid(logits)
    inter = (pred * gt).sum()
    dice  = 1.0 - (2.0 * inter + 1.0) / (pred.sum() + gt.sum() + 1.0)
    return bce + dice


def forward_with_grad(predictor, box_np):
    """
    Replicates predictor._predict without the @torch.no_grad() decorator.
    Requires predictor.set_image() to have been called first.
    Returns (1, 1, H, W) logits.
    """
    # Use _prep_prompts to apply the same box coordinate transform as inference
    _, _, _, unnorm_box = predictor._prep_prompts(
        None, None, box_np, None, normalize_coords=True
    )

    # Convert box to two corner-points with SAM2's box-point labels [2, 3]
    box_coords = unnorm_box.reshape(-1, 2, 2)
    box_labels = torch.tensor([[2, 3]], dtype=torch.int, device=predictor.device)
    box_labels = box_labels.repeat(box_coords.shape[0], 1)

    sparse_emb, dense_emb = predictor.model.sam_prompt_encoder(
        points=(box_coords, box_labels), boxes=None, masks=None
    )

    high_res_features = [
        feat[0].unsqueeze(0) for feat in predictor._features["high_res_feats"]
    ]

    low_res_masks, _, _, _ = predictor.model.sam_mask_decoder(
        image_embeddings=predictor._features["image_embed"][0].unsqueeze(0),
        image_pe=predictor.model.sam_prompt_encoder.get_dense_pe(),
        sparse_prompt_embeddings=sparse_emb,
        dense_prompt_embeddings=dense_emb,
        multimask_output=False,
        repeat_image=False,
        high_res_features=high_res_features,
    )

    return predictor._transforms.postprocess_masks(low_res_masks, predictor._orig_hw[0])

In [ ]:
train_losses, val_losses = [], []
best_val_loss = float("inf")

for epoch in range(epochs):
    # ── Train ──────────────────────────────────────────────────────────────────
    sam2_model.train()
    sam2_model.image_encoder.eval()   # keep BN stats fixed in frozen encoder

    loss_sum, n_steps = 0.0, 0
    shuffled = [train_tiles[i] for i in np.random.permutation(len(train_tiles))]

    for img_tif, label_shp in tqdm(shuffled, desc=f"Epoch {epoch+1}/{epochs} train"):
        instances = load_tile_instances(img_tif, label_shp)
        if not instances:
            continue

        predictor.set_image(instances[0][0])   # encode image once per tile

        optimizer.zero_grad()
        tile_loss = torch.tensor(0.0, device=device)
        for _, box_np, gt_mask in instances:
            tile_loss = tile_loss + seg_loss(forward_with_grad(predictor, box_np)[0, 0], gt_mask)
        (tile_loss / len(instances)).backward()
        optimizer.step()

        loss_sum += tile_loss.item() / len(instances)
        n_steps  += 1

    avg_train = loss_sum / max(n_steps, 1)
    train_losses.append(avg_train)

    # ── Validate ───────────────────────────────────────────────────────────────
    sam2_model.eval()
    loss_sum, n_steps = 0.0, 0

    with torch.no_grad():
        for img_tif, label_shp in tqdm(val_tiles, desc=f"Epoch {epoch+1}/{epochs} val"):
            instances = load_tile_instances(img_tif, label_shp)
            if not instances:
                continue
            predictor.set_image(instances[0][0])
            tile_loss = sum(
                seg_loss(forward_with_grad(predictor, box_np)[0, 0], gt_mask).item()
                for _, box_np, gt_mask in instances
            )
            loss_sum += tile_loss / len(instances)
            n_steps  += 1

    avg_val = loss_sum / max(n_steps, 1)
    val_losses.append(avg_val)
    print(f"Epoch {epoch+1}: train={avg_train:.4f}  val={avg_val:.4f}")

    ckpt_path = ckpt_out_dir / f"sam2_boulder_ep{epoch+1:02d}_val{avg_val:.4f}.pt"
    torch.save(sam2_model.state_dict(), ckpt_path)

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(sam2_model.state_dict(), ckpt_out_dir / "sam2_boulder_best.pt")
        print(f"  → New best: {ckpt_out_dir / 'sam2_boulder_best.pt'}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(train_losses)+1), train_losses, label="train", marker="o")
ax.plot(range(1, len(val_losses)+1),   val_losses,   label="val",   marker="o")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss (BCE + Dice)")
ax.set_title("SAM2 decoder fine-tuning loss")
ax.legend()
plt.tight_layout()
plt.savefig("sam2_finetune_loss.png", dpi=150)
plt.show()